In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import accuracy_score

In [2]:
train =pd.read_csv('./input/train.csv')
test = pd.read_csv('./input/test.csv')
sample_submission = pd.read_csv('./input/sample_submission.csv')

In [ ]:
sample_submission.head()

In [ ]:
train.head(20)

In [ ]:
test.head()

In [3]:
data = pd.concat([train, test], sort=False).reset_index(drop=True)

In [4]:
data["HomePlanet"] = data["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})
data["CryoSleep"] = data["CryoSleep"].fillna(False).map({False: 0, True: 1})
data["Age"] = data["Age"].fillna(data["Age"].median())
data["VIP"] = data["VIP"].fillna(False).map({False: 0, True: 1})
data["Destination"]=data["Destination"].fillna("TRAPPIST-1e").map({"TRAPPIST-1e": 0, "55 Cancri e": 1, "PSO J318.5-22": 2}).astype(int)
data["Payment"] = data[["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]].fillna(0).sum(axis=1)

/tmp/ipykernel_8002/1073139796.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["CryoSleep"] = data["CryoSleep"].fillna(False).map({False: 0, True: 1})
/tmp/ipykernel_8002/1073139796.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data["VIP"] = data["VIP"].fillna(False).map({False: 0, True: 1})


In [5]:
delete_cols = ["Name", "PassengerId", "Cabin", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
data = data.drop(columns=delete_cols)

In [6]:
train = data[:len(train)]
test = data[len(train):]
y_train = train["Transported"].map({False: 0, True: 1})
X_train = train.drop(columns=["Transported"])
X_test = test.drop(columns=["Transported"])
categorical_features = ["HomePlanet", "CryoSleep", "VIP", "Destination"]


In [7]:

y_preds = []
models = []
oof_train = np.zeros((len(X_train),))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

params = {
    'objective': 'binary',
    'max_bin': 300,
    'learning_rate': 0.05,
    'num_leaves': 40
}

for fold_id, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):

    X_tr = X_train.loc[train_index, :]
    X_val = X_train.loc[valid_index, :]
    y_tr = y_train[train_index]
    y_val = y_train[valid_index]
    lgb_train = lgb.Dataset(X_tr, y_tr,
                                             categorical_feature=categorical_features)
    lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train,
                                            categorical_feature=categorical_features)

    model = lgb.train(params, lgb_train,
                                   valid_sets=[lgb_train, lgb_eval],
                                   callbacks=[lgb.log_evaluation(10), lgb.early_stopping(10)],
                                   num_boost_round=1000)
    
    oof_train[valid_index] = model.predict(X_val, num_iteration=model.best_iteration)
    y_pred = model.predict(X_test, num_iteration=model.best_iteration)
    y_preds.append(y_pred)
    models.append(model)

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002690 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 392
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
Training until validation scores don't improve for 10 rounds
[10]	training's binary_logloss: 0.589037	valid_1's binary_logloss: 0.598512
[20]	training's binary_logloss: 0.542578	valid_1's binary_logloss: 0.557907
[30]	training's binary_logloss: 0.51815	valid_1's binary_logloss: 0.539254
[40]	training's binary_logloss: 0.503704	valid_1's binary_logloss: 0.53142
[50]	training's binary_logloss: 0.494629	valid_1's binary_logloss: 0.528015
[60]	train

In [8]:
scores = [
    m.best_score['valid_1']['binary_logloss'] for m in models
]
score = sum(scores) / len(scores)
print('===CV scores===')
print(scores)
print(score)

===CV scores===
[np.float64(0.5275802524779405), np.float64(0.5310039134775602), np.float64(0.5200804278307186), np.float64(0.5308576860576968), np.float64(0.5257748193872707)]
0.5270594198462374


In [9]:
from sklearn.metrics import accuracy_score

y_pred_oof = (oof_train > 0.5).astype(int)
accuracy_score(y_train, y_pred_oof)

0.7407109168296331

In [10]:
y_pred = (y_pred > 0.5).astype(int)
y_pred[:10]


array([1, 0, 1, 0, 0, 0, 1, 1, 1, 0])

In [11]:
y_sub = sum(y_preds) / len(y_preds)
y_sub = (y_sub > 0.5).astype(int)
y_sub[:10]

array([1, 0, 1, 0, 0, 0, 1, 1, 1, 0])

In [ ]:
sub = pd.read_csv('./input/sample_submission.csv')
sub['Transported'] = y_sub.astype(bool)
sub.to_csv('s_t_submission_lightgbm_v2_payment.csv', index=False)

sub.head()
#public score:0.74000→0.74327

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,False
4,0023_01,False
